# Fitting a quadratic by hand

The normal equations on a symmetric design split in two. This lesson solves them for a five-point table whose least-squares quadratic is exactly $1 + 2x + 3x^2$, then checks that the residuals are orthogonal to every column.


## The table

$$
\begin{array}{c|ccccc}
x & -2 & -1 & 0 & 1 & 2 \\
\hline
y & 10 & 0 & 1 & 8 & 16
\end{array}
$$

The right-hand side $X^{\mathsf T}y$ needs three sums. $\sum x = 0$ and $\sum x^3 = 0$ have already been used.

$$
\begin{aligned}
\sum y &= 10 + 0 + 1 + 8 + 16 = 35, \\
\sum xy &= (-2)(10) + (-1)(0) + 0 + (1)(8) + (2)(16) = 20, \\
\sum x^2 y &= 4\cdot 10 + 1\cdot 0 + 0 + 1\cdot 8 + 4\cdot 16 = 112.
\end{aligned}
$$

Because the Gram matrix is block diagonal, the slope equation is alone,

$$
10 b_1 = 20 \Longrightarrow b_1 = 2,
$$

and the even block is

$$
\begin{aligned}
5b_0 + 10b_2 &= 35, \\
10b_0 + 34b_2 &= 112.
\end{aligned}
$$

Divide the first equation by $5$ and the second by $2$:

$$
\begin{aligned}
b_0 + 2b_2 &= 7, \\
5b_0 + 17b_2 &= 56.
\end{aligned}
$$

Substitute $b_0 = 7 - 2b_2$:

$$
5(7 - 2b_2) + 17b_2 = 56 \Longrightarrow 35 + 7b_2 = 56 \Longrightarrow b_2 = 3, \qquad b_0 = 1.
$$

The fitted polynomial is $1 + 2x + 3x^2$. The five predictions of the pure polynomial, computed in the first lesson, were $9, 2, 1, 6, 17$. The residuals against this table are

$$
e = 10 - 9,\ 0 - 2,\ 1 - 1,\ 8 - 6,\ 16 - 17 = 1,\ -2,\ 0,\ 2,\ -1.
$$

Their squares sum to $1 + 4 + 0 + 4 + 1 = 10$.


## Why these residuals had to appear

The normal equations require $X^{\mathsf T}e = 0$. Check the three inner products:

$$
\begin{aligned}
\sum e &= 1 - 2 + 0 + 2 - 1 = 0, \\
\sum xe &= (-2)(1) + (-1)(-2) + 0 + (1)(2) + (2)(-1) = 0, \\
\sum x^2 e &= 4(1) + 1(-2) + 0 + 1(2) + 4(-1) = 0.
\end{aligned}
$$

The residual vector was built to be orthogonal to $1$, $x$, and $x^2$. Adding it to $1 + 2x + 3x^2$ cannot change the least-squares coefficients of a quadratic. The calculus of the loss says the same thing. With

$$
S(b_0, b_1, b_2) = \sum_{i=1}^{5}\bigl(y_i - b_0 - b_1 x_i - b_2 x_i^2\bigr)^2,
$$

the partial derivatives vanish exactly at $(1, 2, 3)$, and the Hessian is twice $X^{\mathsf T}X$, which is positive definite because its determinant is $700$.


In [1]:
# Solve the block equations and match the residual vector 1, -2, 0, 2, -1.
from fractions import Fraction
import sympy as sp
x = [-2, -1, 0, 1, 2]
y = [10, 0, 1, 8, 16]
sum_y = sum(y)
sum_xy = sum(xi * yi for xi, yi in zip(x, y))
sum_x2y = sum(xi ** 2 * yi for xi, yi in zip(x, y))
slope = Fraction(sum_xy, 10)
# b0 + 2 b2 = 7 and 5 b0 + 17 b2 = 56
b2 = Fraction(56 - 35, 7)
b0 = 7 - 2 * b2
fitted = [b0 + slope * xi + b2 * xi ** 2 for xi in x]
errors = [yi - yh for yi, yh in zip(y, fitted)]
print("coefficients", b0, slope, b2)
print("residuals", errors, "RSS", sum(e ** 2 for e in errors))
b0s, b1s, b2s = sp.symbols("b0 b1 b2")
loss = sum((yi - b0s - b1s * xi - b2s * xi ** 2) ** 2 for xi, yi in zip(x, y))
solution = sp.solve(
    [sp.diff(loss, b0s), sp.diff(loss, b1s), sp.diff(loss, b2s)],
    [b0s, b1s, b2s],
)
print("critical point", solution)
assert (b0, slope, b2) == (1, 2, 3)
assert errors == [1, -2, 0, 2, -1]
assert sum(e ** 2 for e in errors) == 10
assert solution == {b0s: 1, b1s: 2, b2s: 3}
assert sum(errors) == 0
assert sum(xi * ei for xi, ei in zip(x, errors)) == 0
assert sum(xi ** 2 * ei for xi, ei in zip(x, errors)) == 0


coefficients 1 2 3
residuals [Fraction(1, 1), Fraction(-2, 1), Fraction(0, 1), Fraction(2, 1), Fraction(-1, 1)] RSS 10
critical point {b0: 1, b1: 2, b2: 3}


## One change: add the same number to every response

Add $4$ to each $y$. Then $\sum y$ grows by $20$, while $\sum xy$ and $\sum x^2 y$ are unchanged only if the added vector is orthogonal to $x$ and to $x^2$. A constant vector is orthogonal to $x$, because $\sum x = 0$. It is not orthogonal to $x^2$:

$$
\sum x^2 \cdot 4 = 40.
$$

The new even equations are $5b_0 + 10b_2 = 55$ and $10b_0 + 34b_2 = 152$. The same elimination gives $b_2 = 3$ still, and $b_0 = 5$. The slope stays $2$. Adding a constant moves only the intercept, here from $1$ to $5$, which is $1 + 4$. The curvature is a contrast among the rows, and a constant has no contrast.


In [2]:
# A constant shift of y changes the intercept and leaves the slope and curvature.
from fractions import Fraction
x = [-2, -1, 0, 1, 2]
y = [14, 4, 5, 12, 20]
sum_xy = sum(xi * yi for xi, yi in zip(x, y))
sum_x2y = sum(xi ** 2 * yi for xi, yi in zip(x, y))
slope = Fraction(sum_xy, sum(xi ** 2 for xi in x))
# 5 b0 + 10 b2 = 55, 10 b0 + 34 b2 = 152
# b0 + 2 b2 = 11, 5 b0 + 17 b2 = 76
b2 = Fraction(76 - 55, 7)
b0 = 11 - 2 * b2
print("shifted coefficients", b0, slope, b2)
assert (b0, slope, b2) == (5, 2, 3)


shifted coefficients 5 2 3


## Pitfall

The predictions $9, 2, 1, 6, 17$ are the fitted mean, not a claim that the next observation at $x = -2$ will be $9$. The observed response there was $10$. The residual $1$ is part of the fit. Quoting $\mu(-2) = 9$ as if the table had said $9$ erases the only evidence this sample has about noise.


In [3]:
# The fitted value and the observation are different numbers.
observed = {(-2): 10, (-1): 0, 0: 1, 1: 8, 2: 16}
fitted = {(-2): 9, (-1): 2, 0: 1, 1: 6, 2: 17}
gaps = [observed[x] - fitted[x] for x in (-2, -1, 0, 1, 2)]
print("observation minus fit", gaps)
assert gaps == [1, -2, 0, 2, -1]
assert observed[-2] != fitted[-2]


observation minus fit [1, -2, 0, 2, -1]


## Summary

- On this symmetric table the least-squares quadratic is exactly $1 + 2x + 3x^2$.
- The residuals are $1, -2, 0, 2, -1$ and $\mathrm{RSS} = 10$. They are orthogonal to $1$, $x$, and $x^2$.
- Adding $4$ to every response moves the intercept from $1$ to $5$ and leaves $b_1 = 2$ and $b_2 = 3$.
